# Build Final Clean SWAN-SF Tensor

This notebook creates a cleaner model-ready tensor by keeping only the core magnetic-field time-series features and removing metadata, quality flags, flare-history columns, and label-like columns from the already standardized aeon tensors.

The train/test split is preserved exactly:

- `X_train_standardized_aeon.npy` stays the training set.
- `X_test_standardized_aeon.npy` stays the test set.
- `y_train.npy` and `y_test.npy` are copied unchanged.

This notebook does **not** randomize, resplit, reshuffle, or refit a scaler. It only removes unwanted channels from the existing tensors.

## 1. Configuration

Update `DATA_DIR` to the folder that contains your current standardized dataset files.

Expected input files:

- `X_train_standardized_aeon.npy`
- `X_test_standardized_aeon.npy`
- `y_train.npy`
- `y_test.npy`
- `feature_columns.json`

Output files will be written to `OUTPUT_DIR`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print('Google Drive mount skipped:', e)

In [ ]:
from pathlib import Path

# CHANGE THIS PATH if your files are somewhere else.
DATA_DIR = Path('/content/drive/MyDrive/solar_flare_forecasting/model_ready_partition_split')

# Output folder for the final clean tensors.
OUTPUT_DIR = DATA_DIR / "final_clean_magnetic_only"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Input file names.
X_TRAIN_PATH = DATA_DIR / "X_train_standardized_aeon.npy"
X_TEST_PATH = DATA_DIR / "X_test_standardized_aeon.npy"
Y_TRAIN_PATH = DATA_DIR / "y_train.npy"
Y_TEST_PATH = DATA_DIR / "y_test.npy"
FEATURE_COLUMNS_PATH = DATA_DIR / "feature_columns.json"

# Output file names.
X_TRAIN_FINAL_PATH = OUTPUT_DIR / "X_train.npy"
X_TEST_FINAL_PATH = OUTPUT_DIR / "X_test.npy"
Y_TRAIN_FINAL_PATH = OUTPUT_DIR / "y_train.npy"
Y_TEST_FINAL_PATH = OUTPUT_DIR / "y_test.npy"
FEATURE_COLUMNS_FINAL_PATH = OUTPUT_DIR / "feature_columns_final_magnetic.json"
SUMMARY_PATH = OUTPUT_DIR / "final_tensor_build_summary.json"

print("Input folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

## 2. Imports and Helper Functions

In [ ]:
import json
import shutil
import numpy as np
from datetime import datetime
from typing import List, Tuple


def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")


def load_feature_columns(path: Path) -> List[str]:
    require_file(path)
    with open(path, "r") as f:
        cols = json.load(f)
    if not isinstance(cols, list):
        raise ValueError("feature_columns.json must contain a list of feature names.")
    return [str(c) for c in cols]


def infer_channel_axis(x_shape: Tuple[int, ...], n_features: int) -> int:
    """Infer which axis corresponds to feature channels.

    Expected aeon shape is usually (n_samples, n_channels, n_timepoints),
    so the channel axis should be 1. This function also supports
    (n_samples, n_timepoints, n_channels) as a fallback.
    """
    if len(x_shape) != 3:
        raise ValueError(f"Expected a 3D tensor, got shape {x_shape}")
    if x_shape[1] == n_features:
        return 1
    if x_shape[2] == n_features:
        return 2
    raise ValueError(
        f"Could not find channel axis. Tensor shape is {x_shape}, "
        f"but feature_columns has length {n_features}."
    )


def save_channel_subset(
    src_path: Path,
    dst_path: Path,
    keep_indices: List[int],
    channel_axis: int,
    chunk_size: int = 10_000,
) -> dict:
    """Save a channel-subset copy of a large .npy tensor using memory mapping."""
    require_file(src_path)
    X_src = np.load(src_path, mmap_mode="r")
    old_shape = X_src.shape

    if channel_axis == 1:
        new_shape = (old_shape[0], len(keep_indices), old_shape[2])
    elif channel_axis == 2:
        new_shape = (old_shape[0], old_shape[1], len(keep_indices))
    else:
        raise ValueError("channel_axis must be 1 or 2")

    X_dst = np.lib.format.open_memmap(
        dst_path,
        mode="w+",
        dtype=X_src.dtype,
        shape=new_shape,
    )

    n_samples = old_shape[0]
    for start in range(0, n_samples, chunk_size):
        stop = min(start + chunk_size, n_samples)
        if channel_axis == 1:
            X_dst[start:stop, :, :] = X_src[start:stop, keep_indices, :]
        else:
            X_dst[start:stop, :, :] = X_src[start:stop, :, keep_indices]
        if start == 0 or stop == n_samples or (start // chunk_size) % 10 == 0:
            print(f"  saved rows {start:,} to {stop:,} / {n_samples:,}")

    X_dst.flush()
    del X_dst

    return {
        "src_path": str(src_path),
        "dst_path": str(dst_path),
        "old_shape": list(old_shape),
        "new_shape": list(new_shape),
        "dtype": str(X_src.dtype),
    }

## 3. Load Current Dataset Metadata

This section checks the existing standardized tensors and reads the current feature-channel list.

In [ ]:
for path in [X_TRAIN_PATH, X_TEST_PATH, Y_TRAIN_PATH, Y_TEST_PATH, FEATURE_COLUMNS_PATH]:
    require_file(path)

feature_cols = load_feature_columns(FEATURE_COLUMNS_PATH)

X_train_mm = np.load(X_TRAIN_PATH, mmap_mode="r")
X_test_mm = np.load(X_TEST_PATH, mmap_mode="r")
y_train = np.load(Y_TRAIN_PATH, mmap_mode="r")
y_test = np.load(Y_TEST_PATH, mmap_mode="r")

channel_axis = infer_channel_axis(X_train_mm.shape, len(feature_cols))
if infer_channel_axis(X_test_mm.shape, len(feature_cols)) != channel_axis:
    raise ValueError("Train and test tensors appear to use different channel axes.")

print("X_train shape:", X_train_mm.shape)
print("X_test shape :", X_test_mm.shape)
print("y_train shape:", y_train.shape)
print("y_test shape :", y_test.shape)
print("Number of feature columns:", len(feature_cols))
print("Inferred channel axis:", channel_axis, "(1 means aeon format: samples, channels, timepoints)")

print("Current feature columns:")
for i, col in enumerate(feature_cols):
    print(f"{i:02d}: {col}")

## 4. Define the Final Feature Policy

For the clean baseline tensor, keep only the core SHARP magnetic-field parameters.

This removes:

- IDs and metadata: `TIMESTAMP`, `HARPNUM`, `NOAA_AR`, etc.
- spatiotemporal/position columns: latitude/longitude, Carrington coordinates, heliocentric angle
- flare-history and target-related columns: `BFLARE`, `CFLARE`, `MFLARE`, `XFLARE`, and `_LOC` versions
- quality flags: `QUALITY`, `IS_TMFI`, `SPEI`, `XRQUALITY`
- disk-integrated X-ray flux: `XR_MAX`
- sparse JSON label annotations: all `_LABEL` columns

These columns can be useful as side-car metadata or for filtering, but they should not be baked into the clean baseline `X` tensor.

In [ ]:
# Core SHARP magnetic-field features to keep.
# These are the main physical time-series features used for magnetic-field-based flare prediction.
MAGNETIC_FEATURES_KEEP = [
    "TOTUSJH",   # Total unsigned current helicity
    "TOTBSQ",    # Total magnitude of Lorentz force proxy
    "TOTPOT",    # Total photospheric magnetic free energy density
    "TOTUSJZ",   # Total unsigned vertical current
    "ABSNJZH",   # Absolute value of net current helicity
    "SAVNCPP",   # Sum of modulus of net current per polarity
    "USFLUX",    # Total unsigned flux
    "TOTFZ",     # Sum of z-component of Lorentz force
    "MEANPOT",   # Mean photospheric magnetic free energy density
    "EPSZ",      # Normalized z-component of Lorentz force
    "SHRGT45",   # Fraction of area with shear angle greater than 45 degrees
    "TOTFY",     # Sum of y-component of Lorentz force
    "MEANSHR",   # Mean shear angle
    "EPSY",      # Normalized y-component of Lorentz force
    "MEANGAM",   # Mean inclination angle
    "MEANGBT",   # Mean gradient of total field
    "MEANGBZ",   # Mean gradient of vertical field
    "MEANGBH",   # Mean gradient of horizontal field
    "MEANJZH",   # Mean current helicity
    "TOTFX",     # Sum of x-component of Lorentz force
    "EPSX",      # Normalized x-component of Lorentz force
    "R_VALUE",   # Total unsigned flux near polarity inversion line
    "MEANJZD",   # Mean vertical current density
    "MEANALP",   # Mean characteristic twist parameter
]

# Use uppercase matching so the notebook is robust to lowercase feature lists.
keep_set = {c.upper() for c in MAGNETIC_FEATURES_KEEP}
feature_cols_upper = [c.upper() for c in feature_cols]

keep_indices = [i for i, c in enumerate(feature_cols_upper) if c in keep_set]
final_feature_cols = [feature_cols[i] for i in keep_indices]
dropped_feature_cols = [c for i, c in enumerate(feature_cols) if i not in keep_indices]

missing_expected_features = [c for c in MAGNETIC_FEATURES_KEEP if c.upper() not in feature_cols_upper]

print("Features kept:", len(final_feature_cols))
for i, col in zip(keep_indices, final_feature_cols):
    print(f"KEEP  old_channel={i:02d}  {col}")

print("Features dropped:", len(dropped_feature_cols))
for col in dropped_feature_cols:
    print("DROP ", col)

if missing_expected_features:
    print("WARNING: These expected magnetic features were not found in your feature list:")
    for col in missing_expected_features:
        print("  -", col)

if len(final_feature_cols) == 0:
    raise ValueError("No magnetic features were found. Check feature_columns.json and feature naming.")

## 5. Optional Sanity Checks

These checks verify that the resulting tensors will still align with the existing labels and that no train/test split changes are being made.

In [ ]:
# Label length checks.
if X_train_mm.shape[0] != y_train.shape[0]:
    raise ValueError(f"Train sample mismatch: X has {X_train_mm.shape[0]}, y has {y_train.shape[0]}")
if X_test_mm.shape[0] != y_test.shape[0]:
    raise ValueError(f"Test sample mismatch: X has {X_test_mm.shape[0]}, y has {y_test.shape[0]}")

# Feature count checks.
if channel_axis == 1:
    expected_train_shape = (X_train_mm.shape[0], len(keep_indices), X_train_mm.shape[2])
    expected_test_shape = (X_test_mm.shape[0], len(keep_indices), X_test_mm.shape[2])
else:
    expected_train_shape = (X_train_mm.shape[0], X_train_mm.shape[1], len(keep_indices))
    expected_test_shape = (X_test_mm.shape[0], X_test_mm.shape[1], len(keep_indices))

print("Expected final X_train shape:", expected_train_shape)
print("Expected final X_test shape :", expected_test_shape)
print("y_train will be copied unchanged:", y_train.shape)
print("y_test will be copied unchanged :", y_test.shape)

## 6. Create the Final Clean Tensors

This writes the new tensor files in chunks so it can handle large arrays without loading everything into RAM at once.

In [ ]:
# Increase chunk_size if you have lots of RAM and want it to run faster.
# Decrease it if Colab runs out of memory.
CHUNK_SIZE = 10_000

print("Saving clean training tensor...")
train_save_info = save_channel_subset(
    src_path=X_TRAIN_PATH,
    dst_path=X_TRAIN_FINAL_PATH,
    keep_indices=keep_indices,
    channel_axis=channel_axis,
    chunk_size=CHUNK_SIZE,
)

print("Saving clean test tensor...")
test_save_info = save_channel_subset(
    src_path=X_TEST_PATH,
    dst_path=X_TEST_FINAL_PATH,
    keep_indices=keep_indices,
    channel_axis=channel_axis,
    chunk_size=CHUNK_SIZE,
)

# Copy y arrays unchanged to the output folder.
shutil.copy2(Y_TRAIN_PATH, Y_TRAIN_FINAL_PATH)
shutil.copy2(Y_TEST_PATH, Y_TEST_FINAL_PATH)

# Save final feature list.
with open(FEATURE_COLUMNS_FINAL_PATH, "w") as f:
    json.dump(final_feature_cols, f, indent=2)

print("Done saving final tensors.")
print("X_train final:", X_TRAIN_FINAL_PATH)
print("X_test final :", X_TEST_FINAL_PATH)
print("y_train     :", Y_TRAIN_FINAL_PATH)
print("y_test      :", Y_TEST_FINAL_PATH)
print("features    :", FEATURE_COLUMNS_FINAL_PATH)

## 7. Validate the Saved Files

This reloads the newly saved tensors with memory mapping and confirms their shapes, label counts, and feature list.

In [ ]:
X_train_final = np.load(X_TRAIN_FINAL_PATH, mmap_mode="r")
X_test_final = np.load(X_TEST_FINAL_PATH, mmap_mode="r")
y_train_final = np.load(Y_TRAIN_FINAL_PATH, mmap_mode="r")
y_test_final = np.load(Y_TEST_FINAL_PATH, mmap_mode="r")

with open(FEATURE_COLUMNS_FINAL_PATH, "r") as f:
    final_cols_check = json.load(f)

print("Final X_train shape:", X_train_final.shape)
print("Final X_test shape :", X_test_final.shape)
print("Final y_train shape:", y_train_final.shape)
print("Final y_test shape :", y_test_final.shape)
print("Final feature count:", len(final_cols_check))

print("y_train class counts:")
unique, counts = np.unique(y_train_final, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

print("y_test class counts:")
unique, counts = np.unique(y_test_final, return_counts=True)
print(dict(zip(unique.tolist(), counts.tolist())))

print("Final feature columns:")
for i, col in enumerate(final_cols_check):
    print(f"{i:02d}: {col}")

## 8. Save a Build Summary

The summary JSON records exactly which columns were kept/dropped and confirms that the train/test split was preserved.

In [ ]:
summary = {
    "created_at": datetime.utcnow().isoformat() + "Z",
    "purpose": "Create final clean magnetic-only SWAN-SF aeon tensors by removing non-feature channels.",
    "split_policy": "Preserved existing train/test split. No reshuffling, random split, or scaler refit was performed.",
    "input_files": {
        "X_train": str(X_TRAIN_PATH),
        "X_test": str(X_TEST_PATH),
        "y_train": str(Y_TRAIN_PATH),
        "y_test": str(Y_TEST_PATH),
        "feature_columns": str(FEATURE_COLUMNS_PATH),
    },
    "output_files": {
        "X_train_final": str(X_TRAIN_FINAL_PATH),
        "X_test_final": str(X_TEST_FINAL_PATH),
        "y_train": str(Y_TRAIN_FINAL_PATH),
        "y_test": str(Y_TEST_FINAL_PATH),
        "feature_columns_final": str(FEATURE_COLUMNS_FINAL_PATH),
        "summary": str(SUMMARY_PATH),
    },
    "channel_axis": channel_axis,
    "train_tensor": train_save_info,
    "test_tensor": test_save_info,
    "features_kept_count": len(final_feature_cols),
    "features_kept": final_feature_cols,
    "features_dropped_count": len(dropped_feature_cols),
    "features_dropped": dropped_feature_cols,
    "missing_expected_magnetic_features": missing_expected_features,
}

with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print("Saved summary to:", SUMMARY_PATH)
print(json.dumps(summary, indent=2)[:3000])